# Proctoring hardening (start)

Task 11, Week 4, Phase 2. The sprint theme is Offer Generation & E-Sign, but the AI/ML
slice of this task is unrelated to offers - it's the start of hardening the assessment
proctoring system, specifically: reduce false positives without letting real cheating
through.

Order of work:
1. Look at the data and why a single-signal rule causes false positives
2. Baseline (the naive rule-based detector - roughly what's already running)
3. Tuned model: logistic regression over all signals + a SAFE/REVIEW/FLAGGED band
4. Metrics: baseline vs tuned, on held-out data
5. The actual evidence - real false positives that got fixed
6. One live demo walkthrough


In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import joblib

from baseline import baseline_flag, FEATURES
from explain import explain_prediction
from evaluate import precision_recall_fpr, evaluate_split, false_positive_audit

pd.set_option('display.max_colwidth', 80)


## 1. The data

Synthetic, standing in for the Week 1 integrity-review labels this task depends on -
one row per 60-second monitoring window, five behaviour archetypes (two honest-but-unusual,
one honest-but-noisy-environment, two genuinely cheating). See `src/data_gen.py` for how
it's built, and note the split is by *student*, not by row - so no student's behaviour
appears in more than one split.


In [ ]:
events_df = pd.read_csv('../data/proctoring_events.csv')
print(events_df.shape)
events_df.groupby('archetype')['label_cheating'].agg(['count', 'mean'])


In [ ]:
events_df['split'].value_counts()


## 2. Baseline: the naive rule-based detector

Flag on any single signal crossing a low bar - this is roughly what a system grows into
if every "can we also flag X" request gets OR'd onto the last one. Individually each rule
is defensible; OR-ing five of them together multiplies the false-positive rate.


In [ ]:
sample = events_df[events_df['archetype'] == 'honest_distracted'].iloc[2]
flagged, triggered = baseline_flag(sample)
print('true label:', 'cheating' if sample.label_cheating else 'honest')
print('baseline flags as cheating:', flagged)
print('because:', triggered)


A student who glanced away for a few seconds, now wearing a cheating flag. This is the problem.

## 3. Tuned model

Logistic regression over all 7 signals together (so no single signal dominates the way it
does in the OR-based baseline), with two probability thresholds tuned on the **validation**
split: anything below the low threshold is SAFE, above the high threshold is FLAGGED, and the
band in between is REVIEW - genuinely ambiguous cases get a human look instead of an automatic
flag. High threshold is picked as the lowest cutoff that still keeps recall >= 0.85 on
validation, so we're not quietly trading away the ability to catch real cheating.


In [ ]:
bundle = joblib.load('../models/proctoring_model.pkl')
print('thresholds: SAFE <', bundle['low_threshold'], '<= REVIEW <', bundle['high_threshold'], '<= FLAGGED')
print()
print('standardised coefficients (bigger = more weight in the decision):')
for f, c in zip(bundle['features'], bundle['model'].coef_[0]):
    print(f'  {f:30s} {c:+.3f}')


In [ ]:
result = explain_prediction({f: sample[f] for f in FEATURES}, bundle)
print('same student, tuned model says:', result['status'], '(score', result['suspicion_score'], ')')
print('reason:', result['reason'])


Same input the baseline flagged - the tuned model correctly calls it SAFE.

## 4. Metrics: baseline vs tuned, held-out


In [ ]:
rows = []
for split_name in ['train', 'val', 'test']:
    split_df = events_df[events_df['split'] == split_name]
    baseline_m, tuned_m, review_rate = evaluate_split(split_df, bundle)
    rows.append({'model': 'baseline_rules', 'split': split_name, 'review_rate': None, **baseline_m})
    rows.append({'model': 'tuned_logreg', 'split': split_name, 'review_rate': review_rate, **tuned_m})

metrics_df = pd.DataFrame(rows)
metrics_df[['model', 'split', 'precision', 'recall', 'fpr', 'review_rate']]


On the held-out **test** split specifically: false-positive rate drops from roughly
0.26 (baseline) to under 0.01 (tuned), while recall barely moves (~0.99 -> ~0.92) - the
model is still catching the real cheating, it's just not punishing honest students for it
anymore.


## 5. The evidence - real false positives, fixed

Every test-split window where the baseline wrongly called an honest student a cheater,
and what the tuned model says instead.


In [ ]:
test_df = events_df[events_df['split'] == 'test']
audit_df = false_positive_audit(test_df, bundle)
print(f"{len(audit_df)} baseline false positives in the test split")
print(f"{int(audit_df['fixed'].sum())} of those are no longer auto-flagged by the tuned model")
audit_df.head(8)


## 6. Live demo walkthrough


In [ ]:
demo = events_df[events_df['archetype'] == 'honest_environment_noise'].iloc[0]
print(f"Session {demo.session_id}, window {demo.window_index}")
print("Raw signals:")
for f in FEATURES:
    print(f'  {f}: {demo[f]}')
print()

flagged, triggered = baseline_flag(demo)
print('BEFORE (baseline):', 'CHEATING' if flagged else 'not flagged', '-', '; '.join(triggered) if triggered else '')

result = explain_prediction({f: demo[f] for f in FEATURES}, bundle)
print(f"AFTER (tuned):     {result['status']}  (score {result['suspicion_score']})")
print(f"  reason: {result['reason']}")


A roommate talking in the next room and a flaky webcam angle used to be enough to get this
student flagged as a cheater. The tuned model recognises it for what it is and lets it through
clean - that's the false-positive reduction this task asked for, on a real (if synthetic)
example, not a hand-picked toy case.
